<a href="https://colab.research.google.com/github/IdrisJunaidAI/ITAI_ML_FirstRepo_IdrisJunaid/blob/main/L11_IdrisJunaid_ITAI1371.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 11: Hyperparameter Tuning and AutoML

**Student:** Idris Junaid  
**Course:** ITAI 1371  
**Lab:** L11  
**Topic:** Hyperparameter Tuning and Automated Machine Learning

## Objective

The purpose of this lab is to optimize a machine learning model by tuning its hyperparameters and to explore how Automated Machine Learning can automate preprocessing, model selection, hyperparameter optimization, and ensemble construction.

This notebook uses the Iris dataset and compares four approaches:

1. A baseline Random Forest model
2. Grid Search with cross validation
3. Random Search with cross validation
4. AutoGluon AutoML

## Part 1: What Are Hyperparameters?

Machine learning models contain two related types of values.

**Model parameters** are learned directly from the training data. Examples include regression coefficients and the split rules learned by a decision tree.

**Hyperparameters** are selected before training begins. They control the model structure or the learning process. Examples include the number of trees in a Random Forest, the maximum depth of each tree, and the minimum number of samples required to split a node.

Hyperparameter tuning systematically tests different settings to identify a configuration that generalizes well to unseen data.

## Part 2: Setup and Baseline Model

In [9]:
# Import the libraries required for data preparation, modeling, and evaluation.
import json
import os
import shutil
import time

# Limit parallel workers so the notebook remains stable on shared systems.
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    train_test_split,
)

# Load the Iris dataset.
iris = load_iris()
X = iris.data
y = iris.target

# Preserve the class proportions in both sets by using stratify=y.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")
print(f"Number of features: {X_train.shape[1]}")

# Train a baseline Random Forest using the default hyperparameters.
baseline_model = RandomForestClassifier(random_state=42)
baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)
accuracy_baseline = accuracy_score(y_test, y_pred_baseline)

print(f"Baseline Random Forest test accuracy: {accuracy_baseline:.2%}")

Training samples: 105
Testing samples: 45
Number of features: 4
Baseline Random Forest test accuracy: 88.89%


## Part 3: Grid Search

Grid Search evaluates every combination in a specified hyperparameter grid. It is transparent and reproducible, but its computational cost grows quickly as more parameters and values are added.

### Task 1: Perform a Grid Search

The search below evaluates three values for `n_estimators` and three values for `max_depth`. This produces nine candidate configurations. Five fold cross validation is used to estimate how well each configuration generalizes.

In [4]:
# Define the complete grid of hyperparameter values.
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [5, 10, None],
}

# Create the GridSearchCV object.
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=1,
    return_train_score=True,
)

# Fit all grid combinations using the training data only.
grid_start = time.perf_counter()
grid_search.fit(X_train, y_train)
grid_runtime = time.perf_counter() - grid_start

# Evaluate the best cross validated model on the untouched test set.
grid_best_model = grid_search.best_estimator_
grid_test_predictions = grid_best_model.predict(X_test)
grid_test_accuracy = accuracy_score(y_test, grid_test_predictions)

print(f"Grid combinations evaluated: {len(grid_search.cv_results_['params'])}")
print(f"Best Grid Search parameters: {grid_search.best_params_}")
print(f"Best Grid Search cross validated accuracy: {grid_search.best_score_:.2%}")
print(f"Best Grid Search test accuracy: {grid_test_accuracy:.2%}")
print(f"Grid Search runtime: {grid_runtime:.2f} seconds")

Grid combinations evaluated: 9
Best Grid Search parameters: {'max_depth': 5, 'n_estimators': 50}
Best Grid Search cross validated accuracy: 95.24%
Best Grid Search test accuracy: 88.89%
Grid Search runtime: 7.93 seconds


## Part 4: Random Search

Random Search samples a fixed number of configurations from a larger hyperparameter space. It usually explores broad spaces more efficiently than Grid Search, although it does not guarantee that every possible combination will be evaluated.

### Task 2: Perform a Random Search

The search below samples ten configurations while tuning the number of trees, maximum tree depth, and minimum number of samples required to split a node.

In [5]:
# Define a broader hyperparameter search space.
param_dist = {
    "n_estimators": [int(x) for x in np.linspace(start=50, stop=500, num=10)],
    "max_depth": [5, 10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
}

# Create the RandomizedSearchCV object.
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    n_jobs=1,
    random_state=42,
    return_train_score=True,
)

# Fit the randomly selected configurations using the training data only.
random_start = time.perf_counter()
random_search.fit(X_train, y_train)
random_runtime = time.perf_counter() - random_start

# Evaluate the best random search model on the untouched test set.
random_best_model = random_search.best_estimator_
random_test_predictions = random_best_model.predict(X_test)
random_test_accuracy = accuracy_score(y_test, random_test_predictions)

print(f"Random configurations evaluated: {len(random_search.cv_results_['params'])}")
print(f"Best Random Search parameters: {random_search.best_params_}")
print(f"Best Random Search cross validated accuracy: {random_search.best_score_:.2%}")
print(f"Best Random Search test accuracy: {random_test_accuracy:.2%}")
print(f"Random Search runtime: {random_runtime:.2f} seconds")

Random configurations evaluated: 10
Best Random Search parameters: {'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 20}
Best Random Search cross validated accuracy: 95.24%
Best Random Search test accuracy: 91.11%
Random Search runtime: 23.90 seconds


## Part 5: Introduction to AutoML with AutoGluon

AutoML automates several steps that would otherwise require many manual experiments. AutoGluon can preprocess the data, train multiple model families, tune candidate models, compare validation performance, and construct an ensemble.

The installation cell below runs only when AutoGluon is missing. In Google Colab, the first installation may take several minutes.

In [6]:
# Install AutoGluon only when it is not already available.
import importlib.util
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")

if importlib.util.find_spec("autogluon") is None:
    %pip install -q autogluon.tabular

from autogluon.tabular import TabularPredictor

print("AutoGluon is available.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 8.9 MB/s eta 0:00:00
AutoGluon is available.


In [7]:
# AutoGluon expects a DataFrame containing both features and the target column.
train_data_ag = pd.DataFrame(X_train, columns=iris.feature_names)
train_data_ag["species"] = y_train

test_data_ag = pd.DataFrame(X_test, columns=iris.feature_names)
test_data_ag["species"] = y_test

# Use a clean temporary folder so this cell can be rerun safely.
automl_model_path = "/tmp/autogluon_lab11_iris"
shutil.rmtree(automl_model_path, ignore_errors=True)

# Train AutoGluon for a maximum of 60 seconds.
automl_start = time.perf_counter()
predictor = TabularPredictor(
    label="species",
    eval_metric="accuracy",
    path=automl_model_path,
).fit(
    train_data=train_data_ag,
    time_limit=60,
    presets="medium_quality",
    hyperparameters={"RF": {}, "XT": {}, "KNN": {}},
    num_cpus=2,
    fit_strategy="sequential",
    verbosity=0,
)
automl_runtime = time.perf_counter() - automl_start

# Compare trained models on the same untouched test set.
leaderboard = predictor.leaderboard(test_data_ag, silent=True)
display_columns = [
    column
    for column in [
        "model",
        "score_test",
        "score_val",
        "pred_time_test",
        "fit_time",
        "stack_level",
    ]
    if column in leaderboard.columns
]

print("AutoGluon leaderboard:")
display(leaderboard[display_columns])

best_automl_model = leaderboard.iloc[0]["model"]
best_automl_test_accuracy = float(leaderboard.iloc[0]["score_test"])

print(f"Best AutoGluon model: {best_automl_model}")
print(f"Best AutoGluon test accuracy: {best_automl_test_accuracy:.2%}")
print(f"AutoGluon runtime: {automl_runtime:.2f} seconds")

AutoGluon leaderboard:


,model,score_test,score_val,pred_time_test,fit_time,stack_level
0,KNeighbors,0.933333,1.000000,0.035053,0.011622,1
1,WeightedEnsemble_L2,0.933333,1.000000,0.041110,0.105445,2
2,RandomForest,0.888889,0.952381,0.126331,1.670203,1
3,ExtraTrees,0.888889,0.952381,0.191688,2.333625,1


Best AutoGluon model: KNeighbors
Best AutoGluon test accuracy: 93.33%
AutoGluon runtime: 13.12 seconds


## Part 6: Experiment Summary

In [8]:
# Create a compact comparison table using the same test set for every approach.
results_summary = pd.DataFrame(
    {
        "Approach": [
            "Baseline Random Forest",
            "Grid Search Random Forest",
            "Random Search Random Forest",
            "AutoGluon best model",
        ],
        "Test Accuracy": [
            accuracy_baseline,
            grid_test_accuracy,
            random_test_accuracy,
            best_automl_test_accuracy,
        ],
        "Search Runtime Seconds": [
            np.nan,
            grid_runtime,
            random_runtime,
            automl_runtime,
        ],
    }
)

results_summary["Test Accuracy"] = results_summary["Test Accuracy"].map(
    lambda value: f"{value:.2%}"
)
results_summary["Search Runtime Seconds"] = results_summary[
    "Search Runtime Seconds"
].map(lambda value: "Not applicable" if pd.isna(value) else f"{value:.2f}")

display(results_summary)

# Save key results temporarily so the completed written answer can be verified.
lab11_results = {
    "baseline_test_accuracy": float(accuracy_baseline),
    "grid_best_params": grid_search.best_params_,
    "grid_cv_accuracy": float(grid_search.best_score_),
    "grid_test_accuracy": float(grid_test_accuracy),
    "random_best_params": random_search.best_params_,
    "random_cv_accuracy": float(random_search.best_score_),
    "random_test_accuracy": float(random_test_accuracy),
    "best_automl_model": str(best_automl_model),
    "best_automl_test_accuracy": float(best_automl_test_accuracy),
}

with open("/tmp/lab11_results.json", "w", encoding="utf-8") as result_file:
    json.dump(lab11_results, result_file, indent=2)

,Approach,Test Accuracy,Search Runtime Seconds
0,Baseline Random Forest,88.89%,Not applicable
1,Grid Search Random Forest,88.89%,7.93
2,Random Search Random Forest,91.11%,23.90
3,AutoGluon best model,93.33%,13.12


## Knowledge Check

### 1. What is the main difference between a model parameter and a hyperparameter?

A model parameter is learned automatically from the training data during model fitting. For example, a regression model learns coefficient values from the observations. A hyperparameter is selected before fitting and controls the model structure or training behavior. In this lab, `n_estimators`, `max_depth`, and `min_samples_split` are hyperparameters because they are chosen before the Random Forest is trained.

### 2. When would you choose Grid Search over Random Search, and vice versa?

I would choose Grid Search when the search space is small, when only a few high impact hyperparameters need to be tested, and when I want an exhaustive and reproducible comparison of every specified combination. I would choose Random Search when the search space is larger, when computational time is limited, or when I want to explore more possible values without evaluating every combination. Random Search can often locate a strong configuration faster, while Grid Search guarantees the best result only within the exact grid that was defined.

### 3. Which AutoGluon model performed best, and what makes AutoML powerful?

**Execution result:** In this execution, the top ranked model on the test leaderboard was **KNeighbors**, with a test accuracy of **93.33%**. The `WeightedEnsemble_L2` model tied the same test accuracy, but KNeighbors appeared first in the leaderboard.

AutoML is powerful because it automates much more than changing one or two hyperparameters. It can prepare the data, compare multiple model families, tune candidate models, rank them with validation data, and combine strong models into an ensemble. This creates a high quality baseline quickly. However, the practitioner must still define the problem, choose an appropriate metric, prevent data leakage, evaluate fairness and interpretability, and validate the final model on data that was not used during model selection.

## Reflection

This lab showed me that model performance depends not only on the algorithm selected but also on the settings used to train it. The baseline Random Forest already performed strongly on the Iris dataset, so tuning did not necessarily create a large increase in test accuracy. This is an important result because tuning should be judged by reliable validation and test evidence rather than by the assumption that a more complicated search must always produce a better model.

The most useful comparison was between Grid Search and Random Search. Grid Search was easy to interpret because it evaluated every combination in the defined grid. Random Search explored an additional hyperparameter and a wider range of values using only ten sampled configurations. This demonstrated the tradeoff between exhaustive coverage and computational efficiency.

AutoGluon demonstrated how AutoML can accelerate experimentation by testing several models and producing a leaderboard. My main takeaway is that AutoML is a decision support tool rather than a replacement for human judgment. In regulated areas such as healthcare and pharmaceutical quality, accuracy alone is not sufficient. Data quality, traceability, model explainability, bias evaluation, and human review remain essential before a model can support a real operational or patient related decision.